# Inflation Shock Momentum — Euro area (Eurostat HICP)

Ports the ISM index to the euro area using Eurostat **HICP-by-COICOP** (the
analogue of the US PCE underlying detail). The momentum **engine is unchanged** —
only the data plumbing differs. Built on `src/ism/eu_pipeline.py` + `eurostat.py`,
using the validated FRED→Eurostat mapping in `config/sources_europe.yaml`.

**Caveats (inherent to EU data):**
- HICP detailed COICOP for the euro-area aggregate starts ~1997-2001, so with a
  120-month window the index begins ~2007-2011 — much shorter than the US 1969+.
- There is **no published author series** to validate against; this is an
  exploratory port. Sanity is judged by economic plausibility (e.g. the 2021-22
  surge and 2023-24 disinflation) and the in-sample forecasting fit.
- A few Eurostat dimension codes (ei_bsco_m indicator, jvs filters) vary by
  vintage; the control builder is defensive and reports what it could fetch.

Requires Eurostat reachable; Brent (FRED) and STOXX Europe 600
(`data/raw/external/stoxx600.csv`) are optional controls.

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_ROOT = Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT/"src"))
try:
    from dotenv import load_dotenv; load_dotenv(REPO_ROOT/".env")
except Exception: pass

from ism.engine import ISMConfig
from ism.eurostat import EurostatClient, hicp_price_panel
from ism import eu_pipeline as eup
from ism import forecasting
from ism.transforms import yoy_inflation
OUT = REPO_ROOT/"outputs"; OUT.mkdir(exist_ok=True)
GEO = "EA20"
client = EurostatClient()
print("Eurostat client ready; geo =", GEO)

## Build the euro-area ISM index

In [ ]:
res, infl, weights = eup.compute_eu_ism(client, geo=GEO, max_digits=4, cfg=ISMConfig())
eu = pd.concat([res.ism, res.s_pos, res.s_neg], axis=1)
eu.columns = ["ISM", "Positive Momentum", "Negative Momentum"]
eu.to_csv(REPO_ROOT/"data"/"processed"/"ism_index_europe.csv")
print("EU ISM months:", res.ism.dropna().shape[0], "| categories:", infl.shape[1])
eu.dropna().tail()

## Figure 1 (EU) — ISM index, components, and HICP inflation

In [ ]:
cp00 = hicp_price_panel(client, geo=GEO)["CP00"]
hicp_yoy = yoy_inflation(cp00).rename("hicp_yoy")

fig,(axA,axB)=plt.subplots(2,1,figsize=(11,8),sharex=True)
axA.plot(res.ism.index, res.ism, color="tab:blue", lw=1.4, label="ISM Index (EU)")
axA.axhline(0,color="grey",lw=.6); axA.set_ylabel("ISM Index")
ax2=axA.twinx(); ax2.plot(hicp_yoy.index, hicp_yoy, color="tab:orange", lw=1.0, label="HICP inflation (12m)")
ax2.set_ylabel("HICP inflation (12m, %)")
axA.set_title("Panel A: Euro-area ISM index and HICP inflation"); axA.legend(loc="upper left")
axB.plot(res.s_pos.index, res.s_pos, color="tab:orange", lw=1.1, label="Positive share $S^+$")
axB.plot(res.s_neg.index, res.s_neg, color="tab:blue", lw=1.1, label="Negative share $S^-$")
axB.set_title("Panel B: components"); axB.legend(loc="upper left")
fig.tight_layout(); fig.savefig(OUT/"figure1_EU.png", dpi=130); plt.show()

## Controls and in-sample forecasts (EU Table 1 analogue)

In [ ]:
# Eurostat-based controls (defensive: reports what it could fetch)
ctrl = eup.build_eu_controls(client, geo=GEO, month_index=res.ism.index)

# Optional global / external controls
try:
    from ism.datasources import FredClient
    brent = FredClient().series("DCOILBRENTEU")  # global Brent (daily) -> monthly
    brent = brent.resample("MS").mean().rename("oil")
    ctrl = ctrl.join(brent.reindex(ctrl.index))
except Exception as e:
    print("Brent oil not added:", e)
try:
    from ism.external_data import load_stoxx_europe_600
    ctrl = ctrl.join(load_stoxx_europe_600().reindex(ctrl.index).rename("stoxx"))
except Exception as e:
    print("STOXX not added (drop data/raw/external/stoxx600.csv to enable):", e)

# Map EU control names to the names forecasting.table1 expects, so we can reuse it.
ren = {"hicp_3m":"pce_3m", "infl_exp_1y":"infl_exp_1y", "vu_ratio":"vu_ratio",
       "oil":"oil_wti", "stoxx":"sp500", "spread_10y_3m":"spread_10y_ffr",
       "recession":"nber_recession"}
ctrl_t = ctrl.rename(columns=ren)
print("controls available:", [c for c in ctrl_t.columns])

tbl = forecasting.table1(hicp_yoy.rename("pce_yoy"), res.ism, res.s_pos, res.s_neg,
                         controls=ctrl_t, horizons=(12,24))
tbl

## Notes
- Category set: COICOP leaves capped at 4 digits (class level); raise `max_digits`
  for finer sub-indices (shorter history) or lower it for a coarser, longer panel.
- This reuses the identical engine, in-sample regression, local-projection and
  LASSO modules as the US side; only `eu_pipeline`/`eurostat` differ.
- Validated mappings and open items are in `config/sources_europe.yaml` and
  `docs/DECISIONS.md`.